# 02 — Tool Calling Agent

**Requires:** `GROQ_API_KEY` in `.env`

---

## What's new here?

In notebook 01 the agent used **hand-written rules** to decide which tool to call.  
Here we replace that rule with an **LLM** — the model reads the user's question and decides.

This is the **ReAct** pattern (Reason + Act):

```
User Question
      │
      ▼
  ┌─────────┐
  │   LLM   │  ← Reason: "I need to calculate something"
  └────┬────┘
       │  tool_call: calculator("15% of 240")
       ▼
  ┌─────────┐
  │  Tool   │  ← Act: run the function
  └────┬────┘
       │  result: "36.0"
       ▼
  ┌─────────┐
  │   LLM   │  ← Observe + answer
  └─────────┘
```

**Concepts covered:**
| Concept | What it means |
|---------|---------------|
| `@tool` decorator | Turns a Python function into an LLM-callable tool |
| `bind_tools()` | Attaches tools to the LLM so it knows what's available |
| `ToolNode` | LangGraph node that executes whichever tool the LLM chose |
| `tools_condition` | Built-in helper that checks if the LLM made a tool call |
| `add_messages` reducer | Appends messages to state instead of overwriting them |

## Step 1 — Install & load environment

In [ ]:
# %pip install langgraph langchain langchain-groq python-dotenv --quiet

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

assert os.getenv("GROQ_API_KEY"), "Add GROQ_API_KEY to your .env file"
print("API key loaded.")

## Step 2 — Define the Tools

The `@tool` decorator does two things:
1. Wraps the function so LangChain can call it
2. Reads the **docstring** to generate the tool description that the LLM sees

> Write clear docstrings — the LLM uses them to decide when to call each tool.

In [ ]:
from langchain_core.tools import tool


@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result.
    Use this for any arithmetic: addition, subtraction, multiplication,
    division, percentages, exponents.
    Example inputs: '15 * 8', '100 / 4', '2 ** 10', '15 / 100 * 240'
    """
    try:
        allowed = set("0123456789+-*/.() ")
        if not all(c in allowed for c in expression):
            return f"Error: invalid characters in '{expression}'"
        return str(eval(expression))  # noqa: S307
    except Exception as e:
        return f"Error: {e}"


@tool
def word_counter(text: str) -> str:
    """Count the number of words, characters, and sentences in a piece of text.
    Use this when asked about text length, word count, or character count.
    """
    words = len(text.split())
    chars = len(text)
    sentences = text.count(".") + text.count("!") + text.count("?")
    return f"Words: {words} | Characters: {chars} | Sentences: {sentences}"


@tool
def unit_converter(value: float, from_unit: str, to_unit: str) -> str:
    """Convert between common units of measurement.
    Supports: km/miles, kg/pounds, celsius/fahrenheit, liters/gallons.
    Example: value=100, from_unit='km', to_unit='miles'
    """
    conversions = {
        ("km", "miles"):       lambda v: v * 0.621371,
        ("miles", "km"):       lambda v: v * 1.60934,
        ("kg", "pounds"):      lambda v: v * 2.20462,
        ("pounds", "kg"):      lambda v: v / 2.20462,
        ("celsius", "fahrenheit"): lambda v: (v * 9 / 5) + 32,
        ("fahrenheit", "celsius"): lambda v: (v - 32) * 5 / 9,
        ("liters", "gallons"): lambda v: v * 0.264172,
        ("gallons", "liters"): lambda v: v / 0.264172,
    }
    key = (from_unit.lower(), to_unit.lower())
    if key not in conversions:
        return f"Unsupported conversion: {from_unit} → {to_unit}"
    result = conversions[key](value)
    return f"{value} {from_unit} = {result:.4f} {to_unit}"


TOOLS = [calculator, word_counter, unit_converter]

print("Tools defined:")
for t in TOOLS:
    print(f"  • {t.name}: {t.description[:60]}...")

## Step 3 — Set Up the LLM

`bind_tools()` tells the LLM what tools exist.  
When the model wants to use one, it returns a **tool_call** instead of plain text.

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0,  # deterministic — no creativity needed for tool selection
)

# Attach tools to the LLM
llm_with_tools = llm.bind_tools(TOOLS)

print(f"LLM: {llm.model_name}")
print(f"Tools bound: {[t.name for t in TOOLS]}")

## Step 4 — Define State & Nodes

We use `add_messages` as the reducer so messages **accumulate** in state  
instead of being replaced on each update.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage, HumanMessage


class AgentState(TypedDict):
    # add_messages appends new messages rather than overwriting
    messages: Annotated[list, add_messages]


SYSTEM_PROMPT = """You are a helpful assistant with access to tools.
Always use a tool when the question involves math, unit conversion, or text analysis.
Be concise in your final answer."""


def call_llm(state: AgentState) -> dict:
    """Send the conversation to the LLM and get its response."""
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


print("State and LLM node defined.")

## Step 5 — Build the Graph

`ToolNode` and `tools_condition` are **built-in LangGraph helpers**:
- `ToolNode` automatically runs whichever tool the LLM called
- `tools_condition` checks if the last message contains a tool call

```
START → call_llm
             │
     tools_condition
             │
      ┌──────┴──────┐
  (tool call)   (no tool call)
      │               │
   tool_node        END
      │
  call_llm  (loop back)
```

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

graph = StateGraph(AgentState)

# Nodes
graph.add_node("call_llm", call_llm)
graph.add_node("tool_node", ToolNode(tools=TOOLS))

# Edges
graph.add_edge(START, "call_llm")

# tools_condition returns "tools" if there's a tool call, otherwise END
graph.add_conditional_edges("call_llm", tools_condition)

# After tool runs → back to LLM (so LLM can see the result and respond)
graph.add_edge("tool_node", "call_llm")

agent = graph.compile()
print("Agent compiled.")

## Step 6 — Ask Questions

A helper function to run any question through the agent and print the conversation.

In [ ]:
def ask(question: str) -> str:
    """Run a question through the agent and return the final answer."""
    print(f"\nQ: {question}")
    print("-" * 50)

    result = agent.invoke({"messages": [HumanMessage(content=question)]})

    # Print each message so we can see the reasoning
    for msg in result["messages"]:
        role = msg.__class__.__name__.replace("Message", "")
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  [{role}] → calling tool: {tc['name']}({tc['args']})")
        elif hasattr(msg, "name") and msg.name:  # ToolMessage
            print(f"  [Tool result] {msg.content}")
        else:
            print(f"  [Final] {msg.content}")

    return result["messages"][-1].content

In [ ]:
# Test 1 — calculator
ask("What is 15% of 240?")

In [ ]:
# Test 2 — unit converter
ask("Convert 100 kilometers to miles.")

In [ ]:
# Test 3 — word counter
ask("How many words are in this sentence: 'The quick brown fox jumps over the lazy dog'?")

In [ ]:
# Test 4 — no tool needed (LLM answers directly)
ask("What is the capital of France?")

In [ ]:
# Test 5 — multi-step: needs two tool calls
ask("If I run 5 miles every day for a week, how many kilometers is that in total?")

## Step 7 — Visualise

In [ ]:
from IPython.display import Image, display

try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    print(agent.get_graph().draw_mermaid())

## Summary

| What you built | What it taught |
|---------------|----------------|
| `@tool` functions | Docstrings are the tool's "manual" for the LLM |
| `bind_tools()` | LLM becomes aware of available tools |
| `ToolNode` | Executes tool calls automatically |
| `tools_condition` | Conditional edge: tool call? → run tool; else → END |
| Loop back to `call_llm` | LLM sees the result and gives a final answer |

---

**Next →** [03 — Two-Agent System](../03-two-agent-system/two_agent_system.ipynb)  
Two agents with different roles talk to each other through shared state.